In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from Bio import Phylo
import torch
from sbi.inference import NPE_C
from sbi.analysis import plot_summary
from sbi.diagnostics import run_sbc, check_sbc
from sbi.analysis.plot import sbc_rank_plot
import sys
sys.path.append('../pysimARG')
from clonal_genealogy import ClonalTree
from newick_to_tree import newick_to_tree
from ClonalOrigin_seq_sim import ClonalOrigin_seq_sim
from homoplasy_index import homoplasy_index
from G4_test import G4_test
from LD_r import LD_r

torch_device = "cpu"

c:\Users\u2008181\likelihood-free\sbi_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
WARNING (pytensor.configdefaults): g++ not available, if using conda: `conda install gxx`
WARNING (pytensor.configdefaults): g++ not detected!  PyTensor will be unable to compile C-implementations and will default to Python. Performance may be severely degraded. To remove this warning, set PyTensor flags cxx to an empty string.


In [2]:
genome_simbac = np.loadtxt("../data/SimBac/genomes_bool.csv", delimiter=",", dtype=bool)

In [3]:
genome_simbac.shape

(10, 100000)

In [4]:
np.random.seed(100)
clonal_tree = ClonalTree(n=10)

# Load phylo tree and convert to ClonalTree format
phylo_tree = Phylo.read("../data/SimBac/clonal_frame.nwk", "newick")
Phylo.draw_ascii(phylo_tree)

edge, node_height = newick_to_tree(phylo_tree)
clonal_tree.edge = edge
clonal_tree.node_height = node_height
clonal_tree.height = np.max(node_height)
clonal_tree.length = np.sum(edge[:, 2])

                                                                   _______ 2
                                        __________________________|
                            ___________|                          |_______ 8
                           |           |
                           |           |___________________________________ 1
                           |
                        ___|                         _____________________ 6
                       |   |                       ,|
                       |   |                       ||  ___________________ 3
                       |   |                       ||_|
                       |   |_______________________|  |___________________ 9
  _____________________|                           |
 |                     |                           |         _____________ 5
 |                     |                           |________|
_|                     |                                    |_____________ 7
 |                  

In [5]:
x_o_500 = np.full((100, 19), np.nan)
x_o_2000 = np.full((100, 19), np.nan)
x_o_6000 = np.full((100, 19), np.nan)

x_o_500, x_o_2000, x_o_6000

(array([[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]], shape=(100, 19)),
 array([[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]], shape=(100, 19)),
 array([[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]], shape=(100, 19)))

In [6]:
np.random.seed(500)
L = 500

for idx in range(100):
    start_pos = np.random.randint(1, genome_simbac.shape[1]-L+1)
    mat = genome_simbac[:, start_pos-1:start_pos+L-1]

    # Identify segregating sites
    has_true = mat.any(axis=0)
    has_false = ~mat.all(axis=0)
    idx_seg = np.where(has_true & has_false)[0]

    # Summary statistics LD r and G4 test
    seg_near, seg_far = 0, 0
    seg_20_50, seg_50_80 = 0, 0
    ld_near, ld_far, g4_near, g4_far = 0, 0, 0, 0
    ld_20_50, ld_50_80, g4_20_50, g4_50_80 = 0, 0, 0, 0
    if idx_seg.size >= 2:
        for i in range(idx_seg.size - 1):
            for j in range(i + 1, idx_seg.size):
                dist_ij = idx_seg[j] - idx_seg[i]
                idx_pair = [idx_seg[i], idx_seg[j]]
                if dist_ij < L/2:
                    ld_near += LD_r(mat[:, idx_pair])
                    g4_near += G4_test(mat[:, idx_pair])
                    seg_near += 1
                else:
                    ld_far += LD_r(mat[:, idx_pair])
                    g4_far += G4_test(mat[:, idx_pair])
                    seg_far += 1
                if 20 <= dist_ij < 50:
                    ld_20_50 += LD_r(mat[:, idx_pair])
                    g4_20_50 += G4_test(mat[:, idx_pair])
                    seg_20_50 += 1
                if 50 <= dist_ij <= 80:
                    ld_50_80 += LD_r(mat[:, idx_pair])
                    g4_50_80 += G4_test(mat[:, idx_pair])
                    seg_50_80 += 1
        
        x_o_500[idx, 0] = ld_near
        x_o_500[idx, 1] = ld_far
        x_o_500[idx, 2] = g4_near
        x_o_500[idx, 3] = g4_far

        x_o_500[idx, 4] = ld_near / seg_near if seg_near > 0 else 0
        x_o_500[idx, 5] = ld_far /seg_far if seg_far > 0 else 0
        x_o_500[idx, 6] = g4_near / seg_near if seg_near > 0 else 0
        x_o_500[idx, 7] = g4_far / seg_far if seg_far > 0 else 0

        x_o_500[idx, 8] = ld_20_50
        x_o_500[idx, 9] = ld_50_80
        x_o_500[idx, 10] = g4_20_50
        x_o_500[idx, 11] = g4_50_80

        x_o_500[idx, 12] = ld_20_50 / seg_20_50 if seg_20_50 > 0 else 0
        x_o_500[idx, 13] = ld_50_80 / seg_50_80 if seg_50_80 > 0 else 0
        x_o_500[idx, 14] = g4_20_50 / seg_20_50 if seg_20_50 > 0 else 0
        x_o_500[idx, 15] = g4_50_80 / seg_50_80 if seg_50_80 > 0 else 0
    else:
        x_o_500[idx, :16] = 0

    # Summary statistic homoplasy index
    x_o_500[idx, 16] = homoplasy_index(clonal_tree, mat)

    # Summary statistic proportion of segregating sites
    count_S = idx_seg.size
    x_o_500[idx, 17] = count_S / L

    # Add the length of sequence as a summary statistic
    x_o_500[idx, 18] = L

    if (idx + 1) % 10 == 0:
        print(f"Completed {idx + 1} out of 100 simulations for L=500")

x_o_500

Completed 10 out of 100 simulations for L=500
Completed 20 out of 100 simulations for L=500
Completed 30 out of 100 simulations for L=500
Completed 40 out of 100 simulations for L=500
Completed 50 out of 100 simulations for L=500
Completed 60 out of 100 simulations for L=500
Completed 70 out of 100 simulations for L=500
Completed 80 out of 100 simulations for L=500
Completed 90 out of 100 simulations for L=500
Completed 100 out of 100 simulations for L=500


array([[4.89236910e+02, 7.90911321e+01, 2.43000000e+02, ...,
        3.47457627e-01, 1.54000000e-01, 5.00000000e+02],
       [1.30225234e+02, 3.84706318e+01, 4.10000000e+01, ...,
        1.89655172e-01, 9.40000000e-02, 5.00000000e+02],
       [1.64287108e+02, 5.46259369e+01, 4.20000000e+01, ...,
        3.04878049e-01, 1.14000000e-01, 5.00000000e+02],
       ...,
       [1.32257964e+02, 3.64943389e+01, 4.10000000e+01, ...,
        2.31884058e-01, 1.06000000e-01, 5.00000000e+02],
       [3.35915497e+02, 1.16553832e+02, 4.57000000e+02, ...,
        5.00000000e-01, 1.52000000e-01, 5.00000000e+02],
       [1.34897042e+02, 3.05163966e+01, 6.10000000e+01, ...,
        4.09638554e-01, 9.80000000e-02, 5.00000000e+02]], shape=(100, 19))

In [7]:
np.random.seed(2000)
L = 2000

for idx in range(100):
    start_pos = np.random.randint(1, genome_simbac.shape[1]-L+1)
    mat = genome_simbac[:, start_pos-1:start_pos+L-1]

    # Identify segregating sites
    has_true = mat.any(axis=0)
    has_false = ~mat.all(axis=0)
    idx_seg = np.where(has_true & has_false)[0]

    # Summary statistics LD r and G4 test
    seg_near, seg_far = 0, 0
    seg_20_50, seg_50_80 = 0, 0
    ld_near, ld_far, g4_near, g4_far = 0, 0, 0, 0
    ld_20_50, ld_50_80, g4_20_50, g4_50_80 = 0, 0, 0, 0
    if idx_seg.size >= 2:
        for i in range(idx_seg.size - 1):
            for j in range(i + 1, idx_seg.size):
                dist_ij = idx_seg[j] - idx_seg[i]
                idx_pair = [idx_seg[i], idx_seg[j]]
                if dist_ij < L/2:
                    ld_near += LD_r(mat[:, idx_pair])
                    g4_near += G4_test(mat[:, idx_pair])
                    seg_near += 1
                else:
                    ld_far += LD_r(mat[:, idx_pair])
                    g4_far += G4_test(mat[:, idx_pair])
                    seg_far += 1
                if 20 <= dist_ij < 50:
                    ld_20_50 += LD_r(mat[:, idx_pair])
                    g4_20_50 += G4_test(mat[:, idx_pair])
                    seg_20_50 += 1
                if 50 <= dist_ij <= 80:
                    ld_50_80 += LD_r(mat[:, idx_pair])
                    g4_50_80 += G4_test(mat[:, idx_pair])
                    seg_50_80 += 1
        
        x_o_2000[idx, 0] = ld_near
        x_o_2000[idx, 1] = ld_far
        x_o_2000[idx, 2] = g4_near
        x_o_2000[idx, 3] = g4_far

        x_o_2000[idx, 4] = ld_near / seg_near if seg_near > 0 else 0
        x_o_2000[idx, 5] = ld_far /seg_far if seg_far > 0 else 0
        x_o_2000[idx, 6] = g4_near / seg_near if seg_near > 0 else 0
        x_o_2000[idx, 7] = g4_far / seg_far if seg_far > 0 else 0

        x_o_2000[idx, 8] = ld_20_50
        x_o_2000[idx, 9] = ld_50_80
        x_o_2000[idx, 10] = g4_20_50
        x_o_2000[idx, 11] = g4_50_80

        x_o_2000[idx, 12] = ld_20_50 / seg_20_50 if seg_20_50 > 0 else 0
        x_o_2000[idx, 13] = ld_50_80 / seg_50_80 if seg_50_80 > 0 else 0
        x_o_2000[idx, 14] = g4_20_50 / seg_20_50 if seg_20_50 > 0 else 0
        x_o_2000[idx, 15] = g4_50_80 / seg_50_80 if seg_50_80 > 0 else 0
    else:
        x_o_2000[idx, :16] = 0

    # Summary statistic homoplasy index
    x_o_2000[idx, 16] = homoplasy_index(clonal_tree, mat)

    # Summary statistic proportion of segregating sites
    count_S = idx_seg.size
    x_o_2000[idx, 17] = count_S / L

    # Add the length of sequence as a summary statistic
    x_o_2000[idx, 18] = L

    if (idx + 1) % 10 == 0:
        print(f"Completed {idx + 1} out of 100 simulations for L=2000")

x_o_2000

Completed 10 out of 100 simulations for L=2000
Completed 20 out of 100 simulations for L=2000
Completed 30 out of 100 simulations for L=2000
Completed 40 out of 100 simulations for L=2000
Completed 50 out of 100 simulations for L=2000
Completed 60 out of 100 simulations for L=2000
Completed 70 out of 100 simulations for L=2000
Completed 80 out of 100 simulations for L=2000
Completed 90 out of 100 simulations for L=2000
Completed 100 out of 100 simulations for L=2000


array([[4.35220020e+03, 1.06134787e+03, 3.32700000e+03, ...,
        4.00809717e-01, 1.48000000e-01, 2.00000000e+03],
       [2.98704881e+03, 9.31960893e+02, 2.64000000e+03, ...,
        4.29928741e-01, 1.20000000e-01, 2.00000000e+03],
       [4.07731604e+03, 1.40888020e+03, 5.04100000e+03, ...,
        4.18604651e-01, 1.37500000e-01, 2.00000000e+03],
       ...,
       [3.42366121e+03, 8.07169240e+02, 3.44000000e+03, ...,
        3.94431555e-01, 1.30500000e-01, 2.00000000e+03],
       [3.69717333e+03, 9.74133648e+02, 2.60000000e+03, ...,
        4.02380952e-01, 1.25500000e-01, 2.00000000e+03],
       [3.30357422e+03, 1.19185043e+03, 1.51800000e+03, ...,
        3.38422392e-01, 1.30000000e-01, 2.00000000e+03]], shape=(100, 19))

In [8]:
np.random.seed(6000)
L = 6000

for idx in range(100):
    start_pos = np.random.randint(1, genome_simbac.shape[1]-L+1)
    mat = genome_simbac[:, start_pos-1:start_pos+L-1]

    # Identify segregating sites
    has_true = mat.any(axis=0)
    has_false = ~mat.all(axis=0)
    idx_seg = np.where(has_true & has_false)[0]

    # Summary statistics LD r and G4 test
    seg_near, seg_far = 0, 0
    seg_20_50, seg_50_80 = 0, 0
    ld_near, ld_far, g4_near, g4_far = 0, 0, 0, 0
    ld_20_50, ld_50_80, g4_20_50, g4_50_80 = 0, 0, 0, 0
    if idx_seg.size >= 2:
        for i in range(idx_seg.size - 1):
            for j in range(i + 1, idx_seg.size):
                dist_ij = idx_seg[j] - idx_seg[i]
                idx_pair = [idx_seg[i], idx_seg[j]]
                if dist_ij < L/2:
                    ld_near += LD_r(mat[:, idx_pair])
                    g4_near += G4_test(mat[:, idx_pair])
                    seg_near += 1
                else:
                    ld_far += LD_r(mat[:, idx_pair])
                    g4_far += G4_test(mat[:, idx_pair])
                    seg_far += 1
                if 20 <= dist_ij < 50:
                    ld_20_50 += LD_r(mat[:, idx_pair])
                    g4_20_50 += G4_test(mat[:, idx_pair])
                    seg_20_50 += 1
                if 50 <= dist_ij <= 80:
                    ld_50_80 += LD_r(mat[:, idx_pair])
                    g4_50_80 += G4_test(mat[:, idx_pair])
                    seg_50_80 += 1
        
        x_o_6000[idx, 0] = ld_near
        x_o_6000[idx, 1] = ld_far
        x_o_6000[idx, 2] = g4_near
        x_o_6000[idx, 3] = g4_far

        x_o_6000[idx, 4] = ld_near / seg_near if seg_near > 0 else 0
        x_o_6000[idx, 5] = ld_far /seg_far if seg_far > 0 else 0
        x_o_6000[idx, 6] = g4_near / seg_near if seg_near > 0 else 0
        x_o_6000[idx, 7] = g4_far / seg_far if seg_far > 0 else 0

        x_o_6000[idx, 8] = ld_20_50
        x_o_6000[idx, 9] = ld_50_80
        x_o_6000[idx, 10] = g4_20_50
        x_o_6000[idx, 11] = g4_50_80

        x_o_6000[idx, 12] = ld_20_50 / seg_20_50 if seg_20_50 > 0 else 0
        x_o_6000[idx, 13] = ld_50_80 / seg_50_80 if seg_50_80 > 0 else 0
        x_o_6000[idx, 14] = g4_20_50 / seg_20_50 if seg_20_50 > 0 else 0
        x_o_6000[idx, 15] = g4_50_80 / seg_50_80 if seg_50_80 > 0 else 0
    else:
        x_o_6000[idx, :16] = 0

    # Summary statistic homoplasy index
    x_o_6000[idx, 16] = homoplasy_index(clonal_tree, mat)

    # Summary statistic proportion of segregating sites
    count_S = idx_seg.size
    x_o_6000[idx, 17] = count_S / L

    # Add the length of sequence as a summary statistic
    x_o_6000[idx, 18] = L

    if (idx + 1) % 10 == 0:
        print(f"Completed {idx + 1} out of 100 simulations for L=6000")

x_o_6000

Completed 10 out of 100 simulations for L=6000
Completed 20 out of 100 simulations for L=6000
Completed 30 out of 100 simulations for L=6000
Completed 40 out of 100 simulations for L=6000
Completed 50 out of 100 simulations for L=6000
Completed 60 out of 100 simulations for L=6000
Completed 70 out of 100 simulations for L=6000
Completed 80 out of 100 simulations for L=6000
Completed 90 out of 100 simulations for L=6000
Completed 100 out of 100 simulations for L=6000


array([[2.84312013e+04, 9.45858658e+03, 2.89170000e+04, ...,
        3.81861575e-01, 1.29500000e-01, 6.00000000e+03],
       [2.64561315e+04, 7.58501511e+03, 2.10210000e+04, ...,
        3.68013758e-01, 1.22500000e-01, 6.00000000e+03],
       [3.04437076e+04, 8.59125506e+03, 3.32450000e+04, ...,
        3.93135725e-01, 1.29666667e-01, 6.00000000e+03],
       ...,
       [4.05275498e+04, 1.28291812e+04, 5.79210000e+04, ...,
        4.21943574e-01, 1.53666667e-01, 6.00000000e+03],
       [2.80704292e+04, 9.12998598e+03, 3.02520000e+04, ...,
        3.94223263e-01, 1.29333333e-01, 6.00000000e+03],
       [2.28071326e+04, 7.58467403e+03, 2.47000000e+04, ...,
        3.93327480e-01, 1.15166667e-01, 6.00000000e+03]], shape=(100, 19))

In [9]:
CO_summary_stats = np.full((3, 19), np.nan)

rho_site = 0.02
theta_site = 0.05
delta = 300
CO_summary_stats

array([[nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan, nan, nan, nan, nan],
       [nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan, nan, nan, nan, nan],
       [nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan, nan, nan, nan, nan]])

In [10]:
np.random.seed(100)
CO_summary_stats[0, :] = ClonalOrigin_seq_sim(clonal_tree, rho_site, theta_site, 500, delta)
CO_summary_stats[1, :] = ClonalOrigin_seq_sim(clonal_tree, rho_site, theta_site, 2000, delta)
CO_summary_stats[2, :] = ClonalOrigin_seq_sim(clonal_tree, rho_site, theta_site, 6000, delta)

CO_summary_stats

array([[9.96053871e+01, 1.41681548e+01, 1.90000000e+01, 2.00000000e+00,
        2.00413254e-01, 1.06527479e-01, 3.82293763e-02, 1.50375940e-02,
        2.08164407e+01, 1.44164541e+01, 4.00000000e+00, 4.00000000e+00,
        2.81303253e-01, 2.21791601e-01, 5.40540541e-02, 6.15384615e-02,
        2.50000000e-01, 7.20000000e-02, 5.00000000e+02],
       [6.13090598e+03, 1.15079756e+03, 5.28100000e+03, 2.48800000e+03,
        1.54899090e-01, 1.00418635e-01, 1.33425973e-01, 2.17102967e-01,
        3.98083919e+02, 3.23600268e+02, 7.00000000e+01, 1.07000000e+02,
        2.60696738e-01, 2.09044101e-01, 4.58415193e-02, 6.91214470e-02,
        4.71074380e-01, 1.60000000e-01, 2.00000000e+03],
       [4.68535740e+04, 1.13617077e+04, 3.41510000e+04, 1.52410000e+04,
        1.37660491e-01, 1.12521121e-01, 1.00339057e-01, 1.50939846e-01,
        1.31776996e+03, 1.13266470e+03, 1.30000000e+02, 2.32000000e+02,
        2.81755391e-01, 2.49540582e-01, 2.77955955e-02, 5.11125799e-02,
        3.31911869e-01

In [11]:
np.savetxt('../data/x_500_mat.csv', x_o_500, delimiter=",")
np.savetxt('../data/x_2000_mat.csv', x_o_2000, delimiter=",")
np.savetxt('../data/x_6000_mat.csv', x_o_6000, delimiter=",")
np.savetxt('../data/x_o_ClonalOrigin.csv', CO_summary_stats, delimiter=",")